In [ ]:
import os  # for file paths
import pandas as pd
import awswrangler as wr
import pydbtools as pydb # see https://github.com/moj-analytical-services/pydbtools
import boto3

# few things for viewing dataframes better
pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 900)
pd.set_option("display.max_colwidth", 200)

In [ ]:

#this is the database we will be extracting from
database = "familyman_live_v4"

#this extracts the latest snapshot from athena
get_snapshot_date = f"SELECT mojap_snapshot_date from {database}.events order by mojap_snapshot_date desc limit 1"
snapshot_date = str(pydb.read_sql_query(get_snapshot_date)['mojap_snapshot_date'].values[0])

#this extracts the February snapshot from athena
#snapshot_date = '2023-02-09'

#this is the athena database we will be storing our tables in
fcsq_database = "fcsq"

#this is the s3 bucket we will be saving data to
s3 = boto3.resource("s3")
bucket = s3.Bucket("alpha-family-data")

#change these to the current quarter and year not the quarter being published
latest_quarter = 4
latest_year = 2023

#change these to the current quarter and year being published
pub_quarter = 3
pub_year = 2023



In [ ]:
# Harm alleged for G50 and G53 events
create_harm_alleged_g50_g53 = f"""
SELECT
  a.field_model,
  a.event,
  b.case_number,
  b.receipt_date,
  b.entry_date
FROM
  {database}.event_fields a
  LEFT JOIN
  {database}.events b
    on a.event = b.event
WHERE
  a.field_model IN ('G50_HA', 'G53_HA')
  AND b.error = 'N'
  AND a.value = 'Y'
  AND a.mojap_snapshot_date = date '{snapshot_date}'
  AND b.mojap_snapshot_date = date '{snapshot_date}'
"""
pydb.create_temp_table(create_harm_alleged_g50_g53, "harm_alleged_g50_g53")

In [ ]:
# Harm alleged for FM2C cases. No event or role recorded so just extracting the case number where the harm alledged value is 'Y'
create_harm_alleged_fm2c = f"""
SELECT
  value,
  field_model,
  case_number
FROM
  {database}.case_fields
WHERE
 field_model = 'FM2C_HA'
 AND value = 'Y'
 AND mojap_snapshot_date = date '{snapshot_date}'
 """

pydb.create_temp_table(create_harm_alleged_fm2c, "harm_alleged_FM2C")

In [ ]:
create_priv_law_case_starts = f"""
SELECT
  Year,
  min_of_Receipt_date as receipt_date,
  Case_number
FROM
  {fcsq_database}.ca_case_starts
WHERE
  main_case_type = 'P'
  AND Year BETWEEN 2011 AND 2022
"""
pydb.create_temp_table(create_priv_law_case_starts, "priv_law_case_starts")

In [ ]:
# Checking Private Law Case Starts
test = pydb.read_sql_query("select year, count(*) as count from __temp__.priv_law_case_starts group by year order by year")
test

In [ ]:
# Add the case start date to the G50/3 HA events so that we can see if they occured at the start of the case or later (add flags for this)
pydb.create_temp_table(f"""
SELECT
  A.*,
  B.Receipt_date as Case_start_date,
  CASE WHEN DATE(A.receipt_date) = DATE(B.receipt_date) 
        THEN 1 ELSE 0
  END AS Start_case,
  CASE WHEN A.receipt_date > B.receipt_date 
       AND B.receipt_date is Not null
        THEN 1 ELSE 0
    END AS During_case
FROM
  __temp__.harm_alleged_g50_g53 A
  LEFT JOIN __temp__.PRIV_LAW_CASE_STARTS B
    ON A.Case_number = B.Case_number
    
    
""",
"gha_dates")                

In [ ]:
# Start of the case
pydb.create_temp_table(f"""
SELECT
  *
FROM
  __temp__.GHA_dates
WHERE
  Start_case = 1
""",
"gha_start")


In [ ]:
#testb = pydb.read_sql_query("select * from gha_start")
#testc = pydb.read_sql_query("select count(*) as count from gha_start")
#testd = pydb.read_sql_query("select count(distinct case_number) as count from gha_start") #7479
#teste = pydb.read_sql_query("select count(*) as count from harm_alleged_g50_g53")
#testb

In [ ]:
# During case
pydb.create_temp_table(f"""
SELECT
  *
FROM
  __temp__.GHA_dates
WHERE
  During_case = 1
""",
"gha_during")

In [ ]:
# Selecting case numbers where harm alleged occurs at the start. Union and select distinct should remove duplicates.
pydb.create_temp_table(f"""
SELECT DISTINCT
  Case_number
FROM
  __temp__.GHA_start
UNION /*not using 'all' eliminates duplicate case numbers.*/
SELECT DISTINCT
  case_number
FROM
  __temp__.harm_alleged_FM2C
""",
"ha_start")

# Selecting case numbers during a case
pydb.create_temp_table(f"""
SELECT DISTINCT
  Case_number
FROM
  __temp__.GHA_during
""",
"ha_during")

In [ ]:
pydb.create_temp_table(f"""
SELECT
  A.Year,
  A.Case_number,
  CASE WHEN S.Case_number IS NULL
       THEN 0 ELSE 1
    END AS HA_Start,
  CASE WHEN D.Case_number IS NULL
       THEN 0 ELSE 1
    END AS HA_During,
  CASE WHEN S.Case_number IS NOT NULL
         OR D.Case_number IS NOT NULL
        THEN 1 ElSE 0
    END AS HA_Any,
  CASE WHEN S.Case_number IS NOT NULL
         AND D.Case_number IS NOT NULL
        THEN 1 ElSE 0
    END AS HA_both_start_during 
FROM
  __temp__.PRIV_LAW_CASE_STARTS A
  LEFT JOIN __temp__.HA_start S
    ON A.Case_number = S.Case_number
  LEFT JOIN __temp__.HA_during D
    ON A.Case_number = D.Case_number
""",
"pl_ha")

In [ ]:
pydb.create_temp_table(f"""
SELECT
  Year,
  SUM(HA_Start) AS Start,
  SUM(HA_During) AS During,
  SUM(HA_Any) AS HA_Total,
  SUM(HA_both_start_during) AS HA_Both,
  COUNT(*) AS Case_starts
FROM  
  __temp__.PL_HA
GROUP BY
  Year
""",
"pl_ha_numbers")

In [ ]:
testb = pydb.read_sql_query("select * from __temp__.pl_ha_numbers order by year")
testb

In [ ]:
testb.to_csv (r's3://alpha-family-data/FOIs/FOI 23112017 - harm_alleged.csv', header = True, index=False)